In [36]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import parselmouth
from praatio import tgio
from parselmouth import PraatError

In [37]:
BASE_DIR = Path("/Users/moanason/Downloads/Data_ANA")

TIER_DIAR_A = "Diarisation_A"
TIER_DIAR_B = "Diarisation_B"
TIER_EVENTS = "TransEvents"
TIER_TRANSCRIBE_A = "Transcribe_A"
TIER_TRANSCRIBE_B = "Transcribe_B"


# PITCH_FLOOR = 75
# PITCH_CEIL = 500

COND_MAP = {
    "NC1": 0, "NC2": 1,
    "NS": 2   # adapt if needed
}

In [ ]:
def parse_session_and_condition(tg_path: Path):
    m = re.search(r"ref_s(\d+)_(N[A-Za-z0-9]+)_processed", tg_path.name)
    if not m:
        raise ValueError(f"Cannot parse session/condition from {tg_path.name}")
    session = int(m.group(1))
    cond_str = m.group(2) 
    if cond_str not in COND_MAP:
        raise ValueError(f"Unknown condition code '{cond_str}' in {tg_path.name}")
    cond_id = COND_MAP[cond_str]
    return session, cond_id, cond_str


def build_audio_paths(session: int, cond_str: str):
    wav_A = BASE_DIR / f"p142_s{session:02d}_A_{cond_str}_processed.wav"
    wav_B = BASE_DIR / f"p142_s{session:02d}_B_{cond_str}_processed.wav"

    return wav_A, wav_B


def load_mono_sound(path: Path):
    snd = parselmouth.Sound(str(path))
    if snd.n_channels > 1:
        snd = snd.convert_to_mono()
    return snd


def mean_f0_and_rms_segment(snd: parselmouth.Sound, t0: float, t1: float):
    seg = snd.extract_part(from_time=t0, to_time=t1, preserve_times=False)

    mean_f0 = np.nan
    
    try:
        pitch = seg.to_pitch()
        
        f0_vals = pitch.selected_array["frequency"]
        voiced_f0_vals = f0_vals[f0_vals > 0]

        if len(voiced_f0_vals) > 0:
            mean_f0 = voiced_f0_vals.mean()

    except PraatError:
        pass

    rms_val = seg.get_rms()

    return mean_f0, rms_val


def count_syllables_in_window(tier, t0: float, t1: float) -> int:
    if tier is None:
        return 0

    count = 0

    if isinstance(tier, tgio.IntervalTier):
        for start, end, label in tier.entryList:
            if end <= t0 or start >= t1:
                continue
            if str(label).strip() != "":
                count += 1

    elif isinstance(tier, tgio.PointTier):
        for time, label in tier.entryList:
            if t0 <= time < t1 and str(label).strip() != "":
                count += 1

    else:
        for entry in tier.entryList:
            if len(entry) == 3:
                start, end, label = entry
                if end <= t0 or start >= t1:
                    continue
                if str(label).strip() != "":
                    count += 1

    return count


def interval_speaking_flag(tier: tgio.IntervalTier, t0: float, t1: float) -> int:
    for start, end, label in tier.entryList:
        # no overlap
        if end <= t0 or start >= t1:
            continue
        # overlap and label is not empty -> speech
        if str(label).strip() != "":
            return 1
    return 0


def classify_event_label(label: str):
    lab = label.strip()

    if lab == "":
        return "Other", None

    # Turn_s##_A/B  -> BET
    if lab.startswith("Turn_"):
        m = re.search(r"_([AB])$", lab)
        if m:
            speaker_main = 0 if m.group(1) == "A" else 1
        else:
            speaker_main = None
        return "BET", speaker_main

    # Backchannel_A/B (or similar)
    if lab.lower().startswith("back"):
        m = re.search(r"_([AB])$", lab)
        if m:
            speaker_main = 0 if m.group(1) == "A" else 1
        else:
            speaker_main = None
        return "Backchannel", speaker_main

    # Overlap / Gap / Silence
    if lab.lower() == "overlap":
        return "Overlap", None
    if lab.lower() == "gap":
        return "Gap", None
    if lab.lower() == "silence":
        return "Silence", None

    # default
    return "Other", None

EVENT_MAP = {
    "BET_A": 10,
    "BET_B": 11,
    "Silence": 20,
    "Gap": 21,
    "Overlap": 30,
    "Backchannel_A": 31,
    "Backchannel_B": 32,
}


def event_code_from_label(etype: str, label_raw: str, speaker_main):
    # Silence / Gap / Overlap don't depend on speaker
    if etype == "Silence":
        return EVENT_MAP["Silence"]
    if etype == "Gap":
        return EVENT_MAP["Gap"]
    if etype == "Overlap":
        return EVENT_MAP["Overlap"]

    # BET: Turn_s##_A/B
    if etype == "BET":
        if speaker_main == 0:
            return EVENT_MAP["BET_A"]
        if speaker_main == 1:
            return EVENT_MAP["BET_B"]
        # if somehow don't know which speaker, fall through to -1

    # Backchannel_A/B
    if etype == "Backchannel":
        if speaker_main == 0:
            return EVENT_MAP["Backchannel_A"]
        if speaker_main == 1:
            return EVENT_MAP["Backchannel_B"]

    # everything else / unknown
    return -1

In [ ]:
def process_textgrid(tg_path: Path):
    """
    For a given combined TextGrid:
      - load separate wav A/B
      - read Diarisation_A/B and TransEvents
      - return list of dict rows for each TransEvents interval:
          TrE types + BET, with F0/RMS per speaker and spk1/spk2 flags.
    """
    print(f"Processing {tg_path.name}")

    session, cond_id, cond_str = parse_session_and_condition(tg_path)
    wav_A_path, wav_B_path = build_audio_paths(session, cond_str)

    snd_A = load_mono_sound(wav_A_path)
    snd_B = load_mono_sound(wav_B_path)

    tg = tgio.openTextgrid(str(tg_path))

    try:
        tier_A = tg.tierDict[TIER_DIAR_A]
        tier_B = tg.tierDict[TIER_DIAR_B]
        tier_events = tg.tierDict[TIER_EVENTS]
        tier_trans_A = tg.tierDict.get(TIER_TRANSCRIBE_A, None)
        tier_trans_B = tg.tierDict.get(TIER_TRANSCRIBE_B, None)
    except KeyError as e:
        raise KeyError(f"Missing tier in {tg_path.name}: {e}")

    rows = []

    for (t0, t1, lab) in tier_events.entryList:
        lab_str = str(lab).strip()
        if t1 <= t0:
            continue  # skip zero-length intervals

        etype, speaker_main = classify_event_label(lab_str)
        e_code = event_code_from_label(etype, lab_str, speaker_main)

        spk1 = interval_speaking_flag(tier_A, t0, t1)
        spk2 = interval_speaking_flag(tier_B, t0, t1)

        # prosody per speaker (only compute if that speaker is active)
        if spk1 == 1:
            F0_A, RMS_A = mean_f0_and_rms_segment(snd_A, t0, t1)
        else:
            F0_A, RMS_A = np.nan, np.nan

        if spk2 == 1:
            F0_B, RMS_B = mean_f0_and_rms_segment(snd_B, t0, t1)
        else:
            F0_B, RMS_B = np.nan, np.nan

        dur = t1 - t0

        # ayllable counts from Transcribe tiers
        n_syl_A = count_syllables_in_window(tier_trans_A, t0, t1)
        n_syl_B = count_syllables_in_window(tier_trans_B, t0, t1)

        # speech rate (syll/s) per speaker
        speechrate_A = n_syl_A / dur if dur > 0 and n_syl_A > 0 else np.nan
        speechrate_B = n_syl_B / dur if dur > 0 and n_syl_B > 0 else np.nan

        if speaker_main == 0:
            speechrate_main = speechrate_A
        elif speaker_main == 1:
            speechrate_main = speechrate_B
        else:
            speechrate_main = np.nan

        row = {
            "id": session,
            "cond": cond_id,
            # "cond_str": cond_str,
            "etype": etype,
            "event": e_code,
            "label": lab_str,
            "spk": speaker_main,   # 0/1/None
            "spk1": spk1,
            "spk2": spk2,
            "n_syl_A": n_syl_A,
            "n_syl_B": n_syl_B,
            "sr_A": speechrate_A,
            "sr_B": speechrate_B,
            "sr": speechrate_main,
            "F0_A": F0_A,
            "RMS_A": RMS_A,
            "F0_B": F0_B,
            "RMS_B": RMS_B,
            "duration": dur,
            "time_sec": t0,
        }

        rows.append(row)

    return rows


In [40]:
all_rows = []

for tg_path in sorted(BASE_DIR.glob("ref_s*_N*_processed.TextGrid")):
    rows = process_textgrid(tg_path)
    all_rows.extend(rows)

trE_df = pd.DataFrame(all_rows)
print("Rows:", len(trE_df))
trE_df.head(20)


Processing ref_s05_NC1_processed.TextGrid
Processing ref_s05_NC2_processed.TextGrid
Processing ref_s06_NC1_processed.TextGrid
Processing ref_s06_NC2_processed.TextGrid
Processing ref_s07_NC1_processed.TextGrid
Processing ref_s07_NC2_processed.TextGrid
Processing ref_s10_NC1_processed.TextGrid
Processing ref_s10_NC2_processed.TextGrid
Processing ref_s11_NC1_processed.TextGrid
Processing ref_s11_NC2_processed.TextGrid
Processing ref_s12_NC1_processed.TextGrid
Processing ref_s12_NC2_processed.TextGrid
Processing ref_s13_NC1_processed.TextGrid
Processing ref_s13_NC2_processed.TextGrid
Processing ref_s16_NC1_processed.TextGrid
Processing ref_s16_NC2_processed.TextGrid
Processing ref_s17_NC1_processed.TextGrid
Processing ref_s17_NC2_processed.TextGrid
Processing ref_s18_NC1_processed.TextGrid
Processing ref_s18_NC2_processed.TextGrid
Processing ref_s19_NC1_processed.TextGrid
Processing ref_s19_NC2_processed.TextGrid
Processing ref_s20_NC1_processed.TextGrid
Processing ref_s20_NC2_processed.T

,id,cond,etype,event,label,spk,spk1,spk2,n_syl_A,n_syl_B,sr_A,sr_B,sr,F0_A,RMS_A,F0_B,RMS_B,duration,time_sec
0,5,0,BET,11,Turn_s05_B,1.0,0,1,0,85,NaN,7.484453,7.484453,NaN,NaN,278.939209,0.006155,11.356875,1.988469
1,5,0,Gap,21,Gap,NaN,0,0,0,1,NaN,2.693603,NaN,NaN,NaN,NaN,NaN,0.371250,13.345344
2,5,0,BET,10,Turn_s05_A,0.0,1,0,17,0,1.869035,NaN,1.869035,123.287822,0.003943,NaN,NaN,9.095602,13.716594
3,5,0,Silence,20,Silence,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.011683,22.812195
4,5,0,BET,10,Turn_s05_A,0.0,1,0,4,0,1.994095,NaN,1.994095,171.726623,0.004715,NaN,NaN,2.005923,23.823878
5,5,0,Silence,20,Silence,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.710608,25.829801
6,5,0,BET,10,Turn_s05_A,0.0,1,0,8,0,2.697893,NaN,2.697893,108.152226,0.003429,NaN,NaN,2.965277,26.540409
7,5,0,Silence,20,Silence,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.343096,29.505686
8,5,0,BET,10,Turn_s05_A,0.0,1,0,7,0,1.782163,NaN,1.782163,103.121416,0.003293,NaN,NaN,3.927812,30.848782
9,5,0,Silence,20,Silence,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.198125,34.776594


In [41]:
out_path = BASE_DIR / "descriptive_processed_TrE_BET_data.csv"
trE_df.to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: /Users/moanason/Downloads/Data_ANA/descriptive_processed_TrE_BET_data.csv


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import parselmouth
from praatio import tgio

# --------------------------
# Paths and naming
# --------------------------
BASE_DIR = Path("/Users/moanason/Downloads/Data_ANA")

TRE_CSV_IN  = BASE_DIR / "descriptive_processed_TrE_BET_data.csv"
TRE_CSV_OUT = BASE_DIR / "descriptive_processed_TrE_BET_IPU_data.csv"

TG_PATTERN = "ref_s*_N*_processed.TextGrid"

COND_MAP = {"NC1": 0, "NC2": 1, "NS": 2}

TIER_DIAR_A   = "Diarisation_A"
TIER_DIAR_B   = "Diarisation_B"
TIER_SYL_A    = "Transcribe_A"
TIER_SYL_B    = "Transcribe_B"
TIER_EVENTS   = "TransEvents"

FRAME_STEP   = 0.01
PITCH_FLOOR  = 75
PITCH_CEIL   = 500

EPS = 1e-6

def parse_id_cond_from_name(stem: str):
    m = re.search(r"ref_s(\d+)_(N[A-Za-z0-9]+)_processed", stem)
    if m is None:
        raise ValueError(f"Cannot parse id/cond from filename stem: {stem}")
    sid = int(m.group(1))
    cond_str = m.group(2)  # e.g. 'NC1'
    if cond_str not in COND_MAP:
        raise ValueError(f"Unknown condition code '{cond_str}' in {stem}")
    cond_num = COND_MAP[cond_str]
    return sid, cond_str, cond_num


def mean_pitch_in_interval(pitch, t0, t1):
    """
    average F0 in [t0, t1] using the Praat pitch object
    """
    vals = []
    t = t0
    while t < t1:
        v = pitch.get_value_at_time(t)
        if np.isfinite(v) and v > 0:
            vals.append(v)
        t += FRAME_STEP
    if len(vals) == 0:
        return np.nan
    return float(np.mean(vals))


def rms_in_interval(sound: parselmouth.Sound, t0: float, t1: float):
    """
    rms of waveform in [t0, t1]
    """
    seg = sound.extract_part(from_time=t0, to_time=t1, preserve_times=False)
    arr = seg.values[0]
    if arr.size == 0:
        return np.nan
    return float(np.sqrt(np.mean(arr ** 2)))


def count_syllables_in_interval(tier, t0, t1):
    if isinstance(tier, tgio.PointTier):
        return sum(
            1 for (time, label) in tier.entryList
            if (t0 <= time < t1) and str(label).strip() != ""
        )
    elif isinstance(tier, tgio.IntervalTier):
        return sum(
            1 for (start, end, label) in tier.entryList
            if (start < t1 and end > t0) and str(label).strip() != ""
        )
    else:
        return 0


def get_backchannel_intervals(events_tier, speaker_label):
    bc_list = []
    suffix = f"_{speaker_label}"
    for (start, end, lab) in events_tier.entryList:
        lab_str = str(lab).strip()
        if lab_str.startswith("Backchannel") and lab_str.endswith(suffix):
            bc_list.append((float(start), float(end)))
    return bc_list


def overlaps_bc(start, end, bc_list):
    for (s, e) in bc_list:
        if (start < e - EPS) and (end > s + EPS):
            return True
    return False


def collect_ipus_from_textgrid(tg_path: Path):
    stem = tg_path.stem
    sid, cond_str, cond_num = parse_id_cond_from_name(stem)

    wav_path = tg_path.with_suffix(".wav")
    if not wav_path.exists():
        raise FileNotFoundError(f"Audio file not found for {tg_path}")

    snd = parselmouth.Sound(str(wav_path))
    if snd.n_channels != 2:
        raise ValueError(f"Expected 2 channels in {wav_path}, found {snd.n_channels}")

    snd_A = snd.extract_channel(1)
    snd_B = snd.extract_channel(2)

    pitch_A = snd_A.to_pitch(time_step=FRAME_STEP,
                             pitch_floor=PITCH_FLOOR,
                             pitch_ceiling=PITCH_CEIL)
    pitch_B = snd_B.to_pitch(time_step=FRAME_STEP,
                             pitch_floor=PITCH_FLOOR,
                             pitch_ceiling=PITCH_CEIL)

    tg = tgio.openTextgrid(str(tg_path))

    diar_A = tg.tierDict[TIER_DIAR_A]
    diar_B = tg.tierDict[TIER_DIAR_B]
    syl_A  = tg.tierDict[TIER_SYL_A]
    syl_B  = tg.tierDict[TIER_SYL_B]
    events = tg.tierDict[TIER_EVENTS]

    # bc per speaker from TransEvents
    bc_A = get_backchannel_intervals(events, "A")
    bc_B = get_backchannel_intervals(events, "B")

    ipu_rows = []

    for (start, end, lab) in diar_A.entryList:
        lab_str = str(lab).strip()
        if lab_str == "":
            continue  # non-speech

        start = float(start)
        end   = float(end)
        dur   = end - start
        if dur <= 0:
            continue

        # exclude diar intervals that overlap Backchannel_A
        if overlaps_bc(start, end, bc_A):
            continue

        n_sylA = count_syllables_in_interval(syl_A, start, end)
        n_sylB = count_syllables_in_interval(syl_B, start, end)

        sr_A = n_sylA / dur if dur > 0 else np.nan
        sr_B = n_sylB / dur if dur > 0 else np.nan

        # main IPU speaker is A (coded spk = 0 here, adapt if needed)
        sr_main = sr_A

        F0_A = mean_pitch_in_interval(pitch_A, start, end)
        RMS_A = rms_in_interval(snd_A, start, end)

        F0_B = mean_pitch_in_interval(pitch_B, start, end)
        RMS_B = rms_in_interval(snd_B, start, end)

        label_ipu = f"IPU_{lab_str}"

        ipu_rows.append(
            dict(
                id       = sid,
                cond     = cond_num,
                etype    = "IPU",
                event    = -1,
                label    = label_ipu,
                spk      = 0,            # 0 = A
                spk1     = 1,            # A speaking
                spk2     = 0,            # B listening
                n_syl_A  = n_sylA,
                n_syl_B  = n_sylB,
                sr_A     = sr_A,
                sr_B     = sr_B,
                sr       = sr_main,
                F0_A     = F0_A,
                RMS_A    = RMS_A,
                F0_B     = F0_B,
                RMS_B    = RMS_B,
                duration = dur,
                time_sec = start,
            )
        )

    for (start, end, lab) in diar_B.entryList:
        lab_str = str(lab).strip()
        if lab_str == "":
            continue

        start = float(start)
        end   = float(end)
        dur   = end - start
        if dur <= 0:
            continue

        if overlaps_bc(start, end, bc_B):
            continue

        n_sylA = count_syllables_in_interval(syl_A, start, end)
        n_sylB = count_syllables_in_interval(syl_B, start, end)

        sr_A = n_sylA / dur if dur > 0 else np.nan
        sr_B = n_sylB / dur if dur > 0 else np.nan

        sr_main = sr_B

        F0_A = mean_pitch_in_interval(pitch_A, start, end)
        RMS_A = rms_in_interval(snd_A, start, end)

        F0_B = mean_pitch_in_interval(pitch_B, start, end)
        RMS_B = rms_in_interval(snd_B, start, end)

        label_ipu = f"IPU_{lab_str}"

        ipu_rows.append(
            dict(
                id       = sid,
                cond     = cond_num,
                etype    = "IPU",
                event    = -1,
                label    = label_ipu,
                spk      = 1,            # 1 = B
                spk1     = 0,            # A listening
                spk2     = 1,            # B speaking
                n_syl_A  = n_sylA,
                n_syl_B  = n_sylB,
                sr_A     = sr_A,
                sr_B     = sr_B,
                sr       = sr_main,
                F0_A     = F0_A,
                RMS_A    = RMS_A,
                F0_B     = F0_B,
                RMS_B    = RMS_B,
                duration = dur,
                time_sec = start,
            )
        )

    return ipu_rows


# load existing TrE+BET data, drop old IPUs
tre_df = pd.read_csv(TRE_CSV_IN)
print("Original rows in TrE+BET:", len(tre_df))

if "etype" not in tre_df.columns:
    raise ValueError("Column 'etype' not found in TrE+BET dataframe.")

tre_no_ipu = tre_df[tre_df["etype"] != "IPU"].copy()
print("Rows after removing old IPU:", len(tre_no_ipu))


# build new IPUs from diarisation tiers (excluding bcs)
all_ipu_rows = []

for tg_path in sorted(BASE_DIR.glob(TG_PATTERN)):
    print("Processing TextGrid for IPUs:", tg_path.name)
    ipu_rows_here = collect_ipus_from_textgrid(tg_path)
    all_ipu_rows.extend(ipu_rows_here)

ipu_df = pd.DataFrame(all_ipu_rows)
print("Number of new IPU rows:", len(ipu_df))

expected_cols = [
    "id", "cond", "etype", "event", "label",
    "spk", "spk1", "spk2",
    "n_syl_A", "n_syl_B",
    "sr_A", "sr_B", "sr",
    "F0_A", "RMS_A", "F0_B", "RMS_B",
    "duration", "time_sec",
]

for col in expected_cols:
    if col not in ipu_df.columns:
        ipu_df[col] = np.nan

ipu_df = ipu_df[expected_cols]


tre_updated = (
    pd.concat([tre_no_ipu, ipu_df], ignore_index=True)
    .sort_values(["id", "cond", "time_sec", "etype"])
    .reset_index(drop=True)
)

tre_updated.to_csv(TRE_CSV_OUT, index=False)
print("Saved updated dataset with correct IPUs (excluding backchannels) to:", TRE_CSV_OUT)


Original rows in TrE+BET: 12560
Rows after removing old IPU: 12560
Processing TextGrid for IPUs: ref_s05_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s05_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s06_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s06_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s07_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s07_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s10_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s10_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s11_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s11_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s12_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s12_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s13_NC1_processed.TextGrid
Processing TextGrid for IPUs: ref_s13_NC2_processed.TextGrid
Processing TextGrid for IPUs: ref_s16_NC1_processed.TextGrid
Processing TextGri